# Development and testing for international crime and prison population visualization

## Script test

In [1]:
%load_ext autoreload
%autoreload 2

## Script development

In [4]:
# Importing libraries
import os

import chart_studio
import chart_studio.plotly as py
import pandas as pd
import plotly.graph_objs as go
import plotly.io as pio
from plotly.subplots import make_subplots
from dotenv import find_dotenv, load_dotenv

from src.visualization import prt_theme

##Loading environment variables
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

##Adding plotly credentials
chart_studio.tools.set_credentials_file(
    username=os.getenv("PLOTLY_USERNAME"), api_key=os.getenv("PLOTLY_API_KEY")
)
#Setting default Plotly template and assigning attributes to prt_template
pio.templates.default = "prt_template"
prt_template = prt_theme.pio.templates['prt_template']

In [5]:
#Read in datasets
df = pd.read_csv("data/processed/sentencing/international_rates.csv")
df

,year,country,value_crime,value_prison
0,1950,England & Wales,952,47
1,1960,England & Wales,1420,59
2,1970,England & Wales,2797,80
3,1980,England & Wales,4772,87
4,1990,England & Wales,7938,90
5,2000,England & Wales,8775,124
6,2010,England & Wales,6613,153
7,2020,England & Wales,6866,133
8,1950,Finland,1272,187
9,1960,Finland,1466,154


In [35]:
fig = make_subplots(cols=3, shared_yaxes=True, specs=[[{"secondary_y": True}, {"secondary_y": True}, {"secondary_y": True}]])
print(fig.layout)

Layout({
    'template': '...',
    'xaxis': {'anchor': 'y', 'domain': [0.0, 0.22444444444444445]},
    'xaxis2': {'anchor': 'y3', 'domain': [0.35777777777777775, 0.5822222222222222]},
    'xaxis3': {'anchor': 'y5', 'domain': [0.7155555555555555, 0.94]},
    'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0]},
    'yaxis2': {'anchor': 'x', 'overlaying': 'y', 'side': 'right'},
    'yaxis3': {'anchor': 'x2', 'domain': [0.0, 1.0], 'matches': 'y', 'showticklabels': False},
    'yaxis4': {'anchor': 'x2', 'overlaying': 'y3', 'side': 'right'},
    'yaxis5': {'anchor': 'x3', 'domain': [0.0, 1.0], 'matches': 'y', 'showticklabels': False},
    'yaxis6': {'anchor': 'x3', 'overlaying': 'y5', 'side': 'right'}
})


In [147]:
# Get the unique values from the 'country' column
titles = df['country'].unique()

# Number of subplots
num_subplots = len(titles)

# Define the gap between subplots
gap = 0.05

# Calculate the width of each subplot
subplot_width = (1 - gap * (num_subplots - 1)) / num_subplots

# Create subplots
fig = make_subplots(
    rows=1, cols=num_subplots, 
    specs=[[{"secondary_y": True}] * num_subplots],
    )

# Set spacing between subplots and add titles
annotations = []

for i in range(num_subplots):
    start_domain = i * (subplot_width + gap)
    end_domain = start_domain + subplot_width
    fig.update_layout(**{f'xaxis{i+1}_domain': [start_domain, end_domain]})

    # Calculate the position for the title
    title_x = (start_domain + end_domain) / 2
    annotations.append(
        dict(
            x=title_x,
            y=1.1,  # This places the title slightly above the plot
            xref="paper",
            yref="paper",
            text=titles[i],
            showarrow=False,
            font=dict(size=14),
            xanchor='center'
        )
    )

# Add the annotations to the layout
fig.update_layout(annotations=annotations)


trace_list= []

for idx, country in enumerate(titles):
    df_country = df[df["country"] == country]

    fig.add_trace(go.Scatter(
        x=df_country["year"],
        y=df_country["value_prison"],
        mode="lines+markers",
        line_color=prt_template.layout.colorway[0],
        text=df_country['country'],
        name="Imprisonment rate",
        hovertemplate="<b>%{text}</b><br>%{x}: %{y} per 100,000"
        ), 
        row=1,
        col=idx+1,
        )
    
    fig.add_trace(go.Scatter(
        x=df_country["year"],
        y=df_country["value_crime"],
        mode="lines+markers",
        line_color=prt_template.layout.colorway[1],
        text=df_country['country'],
        name="Crime rate",
        hovertemplate="<b>%{text}</b><br>%{x}: %{y:,.0f} per 100,000",
        ),
        row=1,
        col=idx+1,
        secondary_y=True)

fig.add_traces(trace_list)

# Set y-axes titles and ranges
fig.update_layout(
    yaxis=dict(
        title_text="Imprisonment rate per 100,000",
        titlefont_color=prt_template.layout.colorway[0],
        ),
    
    yaxis6=dict(
        title_text="Crime rate per 100,000",
        titlefont_color=prt_template.layout.colorway[1],
        ),
)

fig.update_yaxes(
    range=[0,210],
    dtick=50,
    tickfont_color=prt_template.layout.colorway[0],
    secondary_y=False)

fig.update_yaxes(
    range=[0,12600],
    dtick=3000,
    tickformat= ",.0f",
    tickfont_color=prt_template.layout.colorway[1],
    tickmode="sync",
    secondary_y=True)

for i in range(1,3):
    fig.update_yaxes(showticklabels=False, col=i, secondary_y=True)

for i in range(2,4):
    fig.update_yaxes(showticklabels=False, col=i, secondary_y=False)

fig.show()

In [146]:
annotations

[{'x': 0.15,
  'y': 1.1,
  'xref': 'paper',
  'yref': 'paper',
  'text': 'England & Wales',
  'showarrow': False,
  'font': {'size': 14}},
 {'x': 0.49999999999999994,
  'y': 1.1,
  'xref': 'paper',
  'yref': 'paper',
  'text': 'Finland',
  'showarrow': False,
  'font': {'size': 14}},
 {'x': 0.85,
  'y': 1.1,
  'xref': 'paper',
  'yref': 'paper',
  'text': 'Canada',
  'showarrow': False,
  'font': {'size': 14}}]

In [142]:
total = 1
charts = 3
gap = 0.1
chart_space = total/charts - gap

chart_space

0.2333333333333333

In [ ]:
print(fig.layout)

In [ ]:
trace_list[0].name

In [ ]:
for j in range(0, len(trace_list)):
    print(trace_list[j].x[-1])

In [12]:
for idx, country in enumerate(df["country"].unique()):
    # df_country = df[df["country"] == i]
    print(idx, country)

0 England & Wales
1 Finland
2 Canada


In [2]:
from src.visualization.sentencing import plotly_international_rates

In [31]:
fig = plotly_international_rates.prepare_chart()
fig

In [30]:
fig.layout

Layout({
    'annotations': [{'font': {'size': 14},
                     'showarrow': False,
                     'text': '<b>England<br>& Wales</b>',
                     'x': 0.15,
                     'xanchor': 'center',
                     'xref': 'paper',
                     'y': 1.2,
                     'yref': 'paper'},
                    {'font': {'size': 14},
                     'showarrow': False,
                     'text': '<b>Finland</b>',
                     'x': 0.49999999999999994,
                     'xanchor': 'center',
                     'xref': 'paper',
                     'y': 1.2,
                     'yref': 'paper'},
                    {'font': {'size': 14},
                     'showarrow': False,
                     'text': '<b>Canada</b>',
                     'x': 0.85,
                     'xanchor': 'center',
                     'xref': 'paper',
                     'y': 1.2,
                     'yref': 'paper'}],
    'height': 300,
    'ma